In [1]:
import os
import torch
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
class CIFAR10CustomDataset(Dataset):
    def __init__(self, csv_file=None, img_dir=None, transform=None, is_test=False, dataframe=None):
        """Load images from a CSV or a preselected pandas DataFrame.

        Args:
            csv_file (str, optional): Path to CSV annotations.
            img_dir (str): Directory containing images.
            transform (callable, optional): Image preprocessing/augmentation.
            is_test (bool): For unlabeled test data, return IDs instead of labels.
            dataframe (pd.DataFrame, optional): Selected rows for a train/validation split.
                Supply exactly one of csv_file and dataframe.
        """
        if (csv_file is None) == (dataframe is None):
            raise ValueError("Provide exactly one of csv_file or dataframe.")
        self.df = (pd.read_csv(csv_file) if dataframe is None
                   else dataframe.reset_index(drop=True).copy())
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        img_name = self.df.iloc[index]['filename']
        img_path = os.path.join(self.img_dir, img_name)
        with Image.open(img_path) as img:
            image = img.convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            img_id = self.df.iloc[index]['id']
            return image, img_id
        else:
            label = int(self.df.iloc[index]['label'])
            return image, torch.tensor(label, dtype=torch.long)

In [3]:
# --- 1. Transforms & Augmentations ---
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # Apply augmentation only to training images
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

# Deterministic preprocessing for validation and Kaggle test images.
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

# --- 2. Stratified 90/10 Training/Validation Split ---
labels_df = pd.read_csv('./data/train_labels.csv')
train_df, val_df = train_test_split(
    labels_df,
    test_size=0.1,
    random_state=42,
    stratify=labels_df['label'] # training and validation set get the same proportions of each class
)

train_dataset = CIFAR10CustomDataset(
    dataframe=train_df, 
    img_dir='./data/train_images',
    transform=train_transforms, 
    is_test=False
)
val_dataset = CIFAR10CustomDataset(
    dataframe=val_df, 
    img_dir='./data/train_images',
    transform=test_transforms, 
    is_test=False
)
test_dataset = CIFAR10CustomDataset(
    csv_file='./data/test.csv', 
    img_dir='./data/test_images',
    transform=test_transforms, 
    is_test=True
)

# --- 3. Wrap in DataLoaders ---
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f'Training: {len(train_dataset):,} | Validation: {len(val_dataset):,} | '
      f'Kaggle test: {len(test_dataset):,}')

Training: 45,000 | Validation: 5,000 | Kaggle test: 2,000
